In [3]:
import pandas as pd
import os
print(os.getcwd())
df = pd.read_csv("../../../../data/pet/pet_scans_info_covid_controls_ceunim_merged.csv")
output_dir = "../../../../results/pet/paper_analysis/"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

group_col = "group"  # adjust if the group column has a different name
groups = ["LC", "Control"]
numeric_cols = [
    "age_years",
    "height_m",
    "weight_kg",
    "injected_activity_MBq",
    "injected_activity_mCi",
    "glucose_mgdl",
    "time_to_injection_min",
]

df = df[df[group_col].isin(groups)]

# Numeric stats per group
numeric_stats = (
    df.groupby(group_col)[numeric_cols]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .round(2)
)
print("Numeric stats per group:")
display(numeric_stats.T)

# Sex distribution per group
sex_counts = df.groupby(group_col)["sex"].value_counts().unstack(fill_value=0)
sex_percent = (sex_counts.div(sex_counts.sum(axis=1), axis=0) * 100).round(1)
print("Sex distribution (counts):")
display(sex_counts)
print("Sex distribution (%):")
display(sex_percent)


# Now create a table ready to import into a publication with nicely formatted statistics and using mean ± standard deviation for numeric columns
mean_sd_table = (
    df[numeric_cols]
    .agg(["mean", "std"])
    .T
    .assign(
        mean_sd=lambda x: x["mean"].round(2).astype(str)
        + " ± "
        + x["std"].round(2).astype(str)
    )
    [["mean_sd"]]
    .rename(columns={"mean_sd": "Mean ± SD"})
)

mean_sd_table.index.name = "Variable"
print("Mean ± SD table per variable:")
display(mean_sd_table)

group_mean_sd = pd.DataFrame(index=numeric_cols, columns=df[group_col].dropna().unique())

for group in df[group_col].dropna().unique():
    stats = (
        df.loc[df[group_col] == group, numeric_cols]
        .agg(["mean", "std"])
        .T
    )

    group_mean_sd[group] = (
        stats["mean"].round(2).astype(str)
        + " ± "
        + stats["std"].round(2).astype(str)
    )

group_mean_sd.index.name = "Variable"
group_mean_sd.columns.name = group_col

print("Mean ± SD by group:")
display(group_mean_sd)

# Save the tables to the output directory enforcing utf-8 encoding
group_mean_sd.to_csv(os.path.join(output_dir, "demographics_and_acquisition_stats_group_mean_sd.csv"), encoding="utf-8")
mean_sd_table.to_csv(os.path.join(output_dir, "demographics_and_acquisition_stats_mean_sd_table.csv"), encoding="utf-8")

d:\Work\UNSAM\CovidProject\longcovid-neuroimaging-cohort\code\python\pet\paper_analysis
Numeric stats per group:


group                         Control      LC
age_years             count     26.00   29.00
                      mean      63.43   47.03
                      std       15.50    9.77
                      min       24.10   25.00
                      median    69.65   45.00
                      max       78.20   64.00
height_m              count     26.00   29.00
                      mean       1.65    1.65
                      std        0.09    0.10
                      min        1.54    1.47
                      median     1.63    1.64
                      max        1.87    1.86
weight_kg             count     26.00   29.00
                      mean      71.33   82.03
                      std       11.73   18.29
                      min       52.00   56.00
                      median    71.50   80.00
                      max       93.00  119.00
injected_activity_MBq count     26.00   29.00
                      mean     202.62  226.44
                      std       46.43   51.12
                      min      126.54  113.59
                      median   192.47  241.94
                      max      289.71  352.39
injected_activity_mCi count     26.00   29.00
                      mean       5.48    6.12
                      std        1.25    1.38
                      min        3.42    3.07
                      median     5.20    6.54
                      max        7.83    9.52
glucose_mgdl          count     26.00   22.00
                      mean      98.04  100.59
                      std        7.79   14.57
                      min       86.00   86.00
                      median    98.00   97.50
                      max      115.00  155.00
time_to_injection_min count     26.00   29.00
                      mean      73.69   56.93
                      std       31.15   14.19
                      min       26.13   36.62
                      median    79.09   51.40
                      max      161.57   90.63

Sex distribution (counts):


sex,F,M
group,,
Control,16,10
LC,19,10


Sex distribution (%):


sex,F,M
group,,
Control,61.5,38.5
LC,65.5,34.5


Mean ± SD table per variable:


,Mean ± SD
Variable,
age_years,54.79 ± 15.14
height_m,1.65 ± 0.09
weight_kg,76.97 ± 16.32
injected_activity_MBq,215.18 ± 49.97
injected_activity_mCi,5.82 ± 1.35
glucose_mgdl,99.21 ± 11.35
time_to_injection_min,64.85 ± 25.0


Mean ± SD by group:


group,LC,Control
Variable,,
age_years,47.03 ± 9.77,63.43 ± 15.5
height_m,1.65 ± 0.1,1.65 ± 0.09
weight_kg,82.03 ± 18.29,71.33 ± 11.73
injected_activity_MBq,226.44 ± 51.12,202.62 ± 46.43
injected_activity_mCi,6.12 ± 1.38,5.48 ± 1.25
glucose_mgdl,100.59 ± 14.57,98.04 ± 7.79
time_to_injection_min,56.93 ± 14.19,73.69 ± 31.15
